# Universal Day — Hybrid Quantum–Classical Theme Park Optimization

**Hackathon submission / presentation notebook**  
**Focus:** Member 5's data processing, dynamic queue simulation, DFS route ordering, QAOA candidate repair, greedy expansion, and front-end API payload.  

This notebook runs a **real-data classical pipeline immediately** using the included 2026-10-28 recorded queue CSV and ride/walking CSVs. The **actual QAOA section is optional** and needs Member 3's `parkq.py` and `attractions.csv` (not contained in the files provided for this notebook). QAOA is **never silently replaced with synthetic quantum samples**.  

Historical queue estimates are **not live forecasts**. This prototype does not enforce operating hours or queue uncertainty, and assumes no explicit entrance/exit node.

## 1. Architecture and responsibilities

```text
Frontend (Member 1)
    POST /api/optimize: {current_location, preferences, total_time_minutes}
                  ↓
Backend / orchestration
    ├── Member 2: historical queues, walking-time matrix, ride durations
    ├── Member 3: build_qubo(...) [requires parkq.py]
    ├── Member 4: run_parkq_qaoa(...) [quantum candidates]
    └── Member 5:
         data_adapter → route_optimizer (DFS + pruning)
              → time_calculator (arrival-dependent queues)
              → evaluate (remove ≤3, score, greedy insert)
              → integration.plan_to_frontend (JSON)
                  ↓
Frontend displays ordered itinerary
```

To keep the demonstration reproducible and independently runnable, the first experiment uses **the fixed candidate sets explicitly labelled as a classical post-processing demonstration**. A separate section executes genuine QAOA if the two missing Member 3 files are supplied.

In [ ]:
import os, sys, json, itertools, math
from pathlib import Path
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data_adapter.py').exists():
    raise FileNotFoundError('Run the notebook from its extracted package folder alongside the .py modules.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Working folder:', PROJECT_ROOT)
print('Source modules available:', all((PROJECT_ROOT/f).exists() for f in [
    'data_adapter.py', 'time_calculator.py', 'route_optimizer.py',
    'evaluate.py', 'integration.py']))

## 2. Load Member 2's real CSV data

The adapter converts the walking matrix into `walk[from_id][to_id]`, ride durations into `durations[ride_id]['duration']`, and queues into sorted lists of `(datetime, wait_minutes)` pairs. Invalid queue rows are skipped with a warning; no fake waits are created.

In [ ]:
from data_adapter import build_walking_matrix, load_queue_data, load_ride_durations

DATA_DIR = PROJECT_ROOT / 'data'
walking = build_walking_matrix(DATA_DIR / 'walking_time-matrix.csv')
durations = load_ride_durations(DATA_DIR / 'ride_durations.csv')
queue_files = sorted(DATA_DIR.glob('line_up_*.csv'))
if not queue_files:
    raise FileNotFoundError('Place Member 2 line_up_*.csv files in data/.')
queue_data = load_queue_data(queue_files)
ride_ids = sorted(durations)
common_dates = set.intersection(*(
    {timestamp.date() for timestamp, _ in queue_data.get(rid, [])}
    for rid in ride_ids
))
if not common_dates:
    raise ValueError('No historical date is covered for all eight attractions.')
start_time = datetime.combine(max(common_dates), datetime.strptime('09:00', '%H:%M').time())
print('Attractions:', ride_ids)
print('Queue files:', [p.name for p in queue_files])
print('Start time (local park clock):', start_time)
print('Observations:', sum(len(v) for v in queue_data.values()))
pd.DataFrame(walking).T.round(1)

## 3. User preferences and API input

The frontend sends JSON. The integration adapter maps categories to per-attraction scores: `1 + 3 × number of matched tags`. A starting location is a **position**, not a mandatory first ride.

In [ ]:
RIDE_TAGS = {
    'R_1': {'family','immersive'}, 'R_2': {'family','thrill'},
    'R_3': {'magic','immersive','thrill'}, 'R_4': {'magic','family'},
    'R_5': {'thrill','immersive'}, 'R_6': {'thrill','family'},
    'R_7': {'thrill','immersive'}, 'R_8': {'family','immersive'},
}
payload = {
    'current_location': 'R_1',
    'preferences': ['thrill', 'magic'],
    'total_time_minutes': 240,
}
origin = payload['current_location']
budget = payload['total_time_minutes']
preferences = {
    rid: 1.0 + 3.0 * len(tags & set(payload['preferences']))
    for rid, tags in RIDE_TAGS.items()
}
print('Frontend JSON request:', json.dumps(payload, indent=2))
print('Numerical preference scores:', preferences)

## 4. Time-dependent queues and route-order search

The total cost is

$$T=\sum_{i=1}^{n}\left(W_i+Q_i(t_{\mathrm{arrival},i})+D_i\right).$$

Here `W` = walking minutes, `Q` = nearest historical wait on the same date as arrival, and `D` = ride duration. The `route_optimizer.optimize_route_total_time` implementation enumerates ride permutations with **DFS + pruning**: a branch stops when elapsed minutes exceed the budget or are not better than the best complete order. Nonnegative cost increments make that pruning valid.

In [ ]:
from time_calculator import get_queue_time, calculate_total_time
from route_optimizer import optimize_route_total_time

sample = ['R_2', 'R_3', 'R_5']
optimized = optimize_route_total_time(
    sample, walking, durations, queue_data, start_time,
    time_budget=budget, start_location=origin
)
if optimized is None:
    print('No feasible ordering for this example.')
else:
    best_order, info = optimized
    print('Best order:', ' → '.join(best_order))
    print('Walking / queue / ride / total (min):',
          *(round(info[k], 2) for k in ['walking_time','queue_time','ride_time','total_time']))
    display(pd.DataFrame(info['timeline']))

## 5. Candidate repair, preference ranking, and greedy expansion

**Repair:** given a candidate set, enumerate subsets with at most three removed rides; for each subset, find the fastest feasible order.  
**Rank:** prefer higher total preference score; break ties by lower total time, then fewer removed rides.  
**Expand:** after selecting the best repaired QAOA-derived route, try adding each unselected ride, recompute every ordering and dynamic queue, accept the highest-scoring feasible improvement, and repeat. This is *greedy local search*, not a proof of global optimality.

In [ ]:
from evaluate import evaluate_candidates
from integration import plan_to_frontend

# CLASSICAL POST-PROCESSING DEMONSTRATION ONLY: these are specified
# candidate attraction IDs, not sampled QAOA output.
demo_candidates = [
    ['R_1', 'R_3', 'R_5'],
    ['R_2', 'R_4', 'R_8'],
    ['R_3', 'R_5', 'R_6', 'R_7'],
]

def postprocess(candidates, time_budget, expand=True):
    return evaluate_candidates(
        candidates=candidates,
        ride_data=durations, walking_matrix=walking, queue_data=queue_data,
        preferences=preferences, time_budget=time_budget,
        start_time=start_time, max_removals=3,
        start_location=origin, expand_candidates=expand,
    )

without_expansion = postprocess(demo_candidates, budget, expand=False)
with_expansion = postprocess(demo_candidates, budget, expand=True)
for label, result in [('Before expansion', without_expansion), ('After expansion', with_expansion)]:
    print('\n', label, 'status =', result['status'])
    if result['status']=='SUCCESS':
        p = result['best_plan']
        print('  route:', ' → '.join(p['route']))
        print('  rides:', len(p['route']), 'total min:', round(p['total_time'], 2),
              'preference score:', p['preference_score'])
        print('  added:', p.get('added_rides', []), 'removed:', p.get('removed_rides', []))

## 6. Inspect the itinerary and visualize time components

Visualization uses the actual per-stop timeline generated by the classical optimizer; it does not infer any walking distance in metres.

In [ ]:
if with_expansion['status'] == 'SUCCESS':
    plan = with_expansion['best_plan']
    timeline = pd.DataFrame(plan['timeline'])
    display(timeline)
    components = timeline.set_index('ride_id')[['walking_time','queue_time','ride_time']]
    ax = components.plot(kind='bar', stacked=True, figsize=(10, 4))
    ax.set_title('Per-attraction time breakdown (recorded historical queue data)')
    ax.set_xlabel('Ride')
    ax.set_ylabel('Minutes')
    ax.legend(['Walking','Queue','Ride'])
    plt.tight_layout()
    plt.show()
else:
    print('No feasible route to visualize.')

## 7. Integration: exact JSON payload returned to the website

`integration.plan_to_frontend` produces the backend-facing `steps` list consumed by Member 1's JavaScript. `walk_distance_m` and `origin_distance_m` are `null`, because the provided data contains walking time, not measured metres.

In [ ]:
result_json = plan_to_frontend(with_expansion)
result_json.update({
    'solver_mode': 'classical-postprocessing-demo',
    'queue_source': 'historical',
    'queue_data_date': start_time.date().isoformat(),
    'starting_location': origin,
})
print(json.dumps(result_json, ensure_ascii=False, indent=2))
assert result_json.get('status') == 'SUCCESS'
assert all(step['walk_distance_m'] is None for step in result_json['steps'])

## 8. Budget sensitivity test (same user preferences)

How much does a larger time budget help? In this section we keep preferences, origin, CSV inputs, and *demo candidates* unchanged, and compare candidate expansion for several budgets. Results are recorded classical-model outputs, **not a quantum advantage benchmark**.

In [ ]:
budget_tests = [100, 180, 240, 400, 700]
rows=[]
for b in budget_tests:
    result = postprocess(demo_candidates, b, expand=True)
    if result['status'] == 'SUCCESS':
        p=result['best_plan']
        rows.append({'budget_min':b, 'rides':len(p['route']),
                     'total_min':round(p['total_time'],1),
                     'unused_min':round(b-p['total_time'],1),
                     'preference_score':p['preference_score'], 'status':'SUCCESS'})
    else:
        rows.append({'budget_min':b,'rides':0,'total_min':None,
                     'unused_min':None,'preference_score':None,'status':result['status']})
budget_results=pd.DataFrame(rows)
display(budget_results)
ax=budget_results.plot(x='budget_min',y='rides',marker='o',legend=False,figsize=(8,4))
ax.set_ylabel('Number of attractions')
ax.set_xlabel('Budget (minutes)')
ax.set_title('Feasible attraction count vs time budget')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 9. Optional: real QUBO + QAOA from Members 3 and 4

**Required first:** copy `parkq.py` and `attractions.csv` from the team's final project into the same folder as this notebook. Install `qiskit`, `numpy`, and `scipy`. Without those files/dependencies this cell reports a clear skip; it does **not** fabricate measured quantum results.

Member 4's QAOA code uses a fixed initial parameter vector `x0=[0.5,0.5]`, and the updated `StatevectorSampler(seed=42)` for repeatability. This only makes the measurement reproducible under compatible versions/configurations; it does not guarantee globally optimal routes.

**Mac crash note:** this optional cell executes QAOA inside the notebook kernel. If your local Qiskit simulator has shown a native segmentation fault, **skip this cell during a live presentation** and run your separately isolated `safe_optimizer.py` server instead.

In [ ]:
import importlib.util
has_parkq = (PROJECT_ROOT/'parkq.py').is_file()
has_attractions = (PROJECT_ROOT/'attractions.csv').is_file()
has_qiskit = importlib.util.find_spec('qiskit') is not None
RUN_LIVE_QAOA = False  # Change to True ONLY after installing deps and adding the two source files.
print('Member 3 files present:', has_parkq, has_attractions)
print('Qiskit installed:', has_qiskit)
print('RUN_LIVE_QAOA:', RUN_LIVE_QAOA)

qaoa_result = None
if RUN_LIVE_QAOA:
    if not (has_parkq and has_attractions and has_qiskit):
        raise RuntimeError('QAOA requires parkq.py, attractions.csv, and qiskit.')
    from parkq import load_rides
    from parkq_qaoa import run_parkq_qaoa
    rides = load_rides(demo=True)
    for ride in rides:
        ride['ride_minutes'] = durations[ride['ride_id']]['duration']
        if 'duration_time_min' in ride:
            ride['duration_time_min'] = durations[ride['ride_id']]['duration']
    qaoa_result = run_parkq_qaoa(
        rides=rides, preferences=preferences, budget=budget,
        shots=64, top_k=3, maxiter=5, seed=42,
    )
    print('QAOA candidate counts:', len(qaoa_result['candidates']))
else:
    print('Skipped live QAOA safely. Real QAOA results have not been generated in this notebook run.')

## 10. Optional: pass actual QAOA samples through Member 5

The only bridge needed is `extract_qaoa_candidates`, which maps each sampled record's `selected_ids` into the classical pipeline. No fake quantum samples are substituted.

In [ ]:
from evaluate import extract_qaoa_candidates
if qaoa_result is not None:
    sampled_sets = extract_qaoa_candidates(qaoa_result)
    quantum_refined = postprocess(sampled_sets, budget, expand=True)
    ui = plan_to_frontend(quantum_refined)
    ui.update({'solver_mode':'qaoa', 'queue_source':'historical'})
    print('QAOA candidates:', sampled_sets)
    print(json.dumps(ui, indent=2))
else:
    print('Skipped: there is no actual QAOA run to post-process.')

## 11. Conclusions and limitations

- **Member 5's contribution:** CSV preprocessing; arrival-dependent queue simulation; DFS ordering and safe pruning; removing up to three rides when needed; preference-first evaluation; greedy expansion to use otherwise unused budget; UI-compatible JSON.
- **Role of quantum:** QAOA generates candidate attraction subsets from Member 3's QUBO formulation. Classical post-processing may remove or **add** attractions, so the final route need not equal a raw sampled QAOA subset.
- **Limitations:** eight rides only; nearest historical wait instead of forecasting live queues; no park-hours constraints; small `shots/maxiter` may limit QAOA quality; greedy expansion is not globally optimal. Fixed seed controls reproducibility, **not** optimization quality.
- **Reproducibility:** the packaged CSVs are dated historical observations; the default cells are independently runnable without Qiskit, but do **not** demonstrate successful QAOA execution unless the optional cells are enabled with authentic Member 3 inputs.

**Presentation takeaway:** *QAOA generates candidate ride combinations. Member 5 transforms them into feasible, personalized, time-aware itineraries and returns structured results to the web frontend.*